# Taylor Approximation Tests: F0, F1, F2, F3

Verifies the second-order Taylor expansion for each functional in the cascade:
$$f(x + \varepsilon\,dx) = f(x) + \varepsilon\,df(x,dx) + \tfrac{1}{2}\varepsilon^2\,d^2f(x,dx,dx) + O(\varepsilon^3)$$

where $d^2f = \text{d2F\_dF}_k(x, dx, dx,\, w{=}\text{None})$ (pure second-order term, no cascade $w$ feed-through).

For each functional $F_k$:
- **e1** = first-order residual $\|F_k(x+\varepsilon\,dx) - F_k(x) - \varepsilon\,dF_k\|$ — expected $O(\varepsilon^2)$, ratio $\approx 4$ per halving
- **e2** = second-order residual $\|\cdots - \tfrac{1}{2}\varepsilon^2\,d^2F_k\|$ — expected $O(\varepsilon^3)$, ratio $\approx 8$ per halving

**Note:** F1 is bilinear, so e2 is at machine precision (exact second-order approximation).

In [ ]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from mpi4py import MPI
from types import SimpleNamespace
from holotomocupy.rec_mpi import Rec

n      = 16
ntheta = 4
ndist  = 2
nz     = n
nzobj  = n
nobj   = n

args = SimpleNamespace(
    energy                  = 17.1,
    detector_pixelsize      = 1.4760147601476e-6 * 16,
    focustodetectordistance = 1.217,
    z1                      = np.array([5.110, 5.464]) * 1e-3,
    theta                   = np.linspace(0, np.pi, ntheta, dtype='float32'),
    ndist                   = ndist,
    ntheta                  = ntheta,
    nz                      = nz,
    n                       = n,
    nzobj                   = nzobj,
    nobj                    = nobj,
    obj_dtype               = 'complex64',
    mask                    = 0.9,
    lam_prbfit              = 0.0,
    rho                     = [1, 1, 1],
    niter                   = 1,
    nchunk                  = ntheta,
    vis_step                = -1,
    err_step                = -1,
    start_iter              = 0,
    comm                    = MPI.COMM_WORLD,
)

cl = Rec(args)

TypeError: data type 'compelx64' not understood

In [ ]:
# Random evaluation points and perturbation directions
rng = np.random.default_rng(42)

def rc(shape):  # random complex64
    return (rng.standard_normal(shape) + 1j * rng.standard_normal(shape)).astype('complex64')

def rf(shape, scale=1.0):  # random float32
    return (rng.standard_normal(shape) * scale).astype('float32')

prb_g   = cp.array(rc((ndist, nz, n)))
proj_g  = cp.array(rf((ntheta, nzobj, nobj), 0.1))   # small phase (radians)
pos_g   = cp.array(rf((ntheta, ndist, 2),   0.5))    # sub-pixel shifts
data_g  = cp.array(np.abs(rf((ntheta, ndist, nz, n))) + 0.1).astype('float32')  # positive

dprb_g  = cp.array(rc((ndist, nz, n)))
dproj_g = cp.array(rf((ntheta, nzobj, nobj), 0.1))
dpos_g  = cp.array(rf((ntheta, ndist, 2),   0.5))

epss = np.array([0.5, 0.25, 0.125, 0.0625])

## F3:  $(x_{31}, x_{32}, x_{33}) \mapsto [x_{31},\; \mathcal{S}_{x_{32}}(x_{33})]$

Only the second (nonlinear) component is tested.

In [ ]:
x31, x32, x33   = prb_g, pos_g, proj_g
dx31, dx32, dx33 = dprb_g, dpos_g, dproj_g

_, x22_0 = cl.F3([x31, x32, x33])

e1s_F3, e2s_F3 = [], []
for eps in epss:
    _, x22_e  = cl.F3([x31 + eps*dx31, x32 + eps*dx32, x33 + eps*dx33])
    _, df3    = cl.dF3([x31, x32, x33], [dx31, dx32, dx33])
    d2f3      = cl.d2F_dF3([x31, x32, x33], [dx31, dx32, dx33], [dx31, dx32, dx33], [None, None, None])

    res1 = x22_e - x22_0 - eps * df3[1]
    res2 = res1 - 0.5 * eps**2 * d2f3[1]
    e1s_F3.append(float(cp.linalg.norm(res1)))
    e2s_F3.append(float(cp.linalg.norm(res2)))

e1s_F3, e2s_F3 = np.array(e1s_F3), np.array(e2s_F3)

## F2:  $[x_{21}, x_{22}] \mapsto [x_{21},\; e^{i x_{22}}]$

Only the second (nonlinear) component is tested.

In [ ]:
x21_f2  = prb_g.copy()
x22_f2  = cp.array(rf((ntheta, ndist, nz, n), 0.1))
dx21_f2 = dprb_g.copy()
dx22_f2 = cp.array(rf((ntheta, ndist, nz, n), 0.1))

_, x12_0 = cl.F2([x21_f2, x22_f2])

e1s_F2, e2s_F2 = [], []
for eps in epss:
    _, x12_e = cl.F2([x21_f2 + eps*dx21_f2, x22_f2 + eps*dx22_f2])
    _, df2   = cl.dF2([x21_f2, x22_f2], [dx21_f2, dx22_f2])
    d2f2     = cl.d2F_dF2([x21_f2, x22_f2], [dx21_f2, dx22_f2], [dx21_f2, dx22_f2], [None, None])

    res1 = x12_e - x12_0 - eps * df2[1]
    res2 = res1 - 0.5 * eps**2 * d2f2[1]
    e1s_F2.append(float(cp.linalg.norm(res1)))
    e2s_F2.append(float(cp.linalg.norm(res2)))

e1s_F2, e2s_F2 = np.array(e1s_F2), np.array(e2s_F2)

## F1:  $[x_{11}, x_{12}] \mapsto D(x_{11} \cdot x_{12})$

F1 is **bilinear**, so the Taylor expansion is exact at second order: e2 is at machine precision.

In [ ]:
x11_f1  = prb_g.copy()
x12_f1  = cp.array(rc((ntheta, ndist, nz, n)))
dx11_f1 = dprb_g.copy()
dx12_f1 = cp.array(rc((ntheta, ndist, nz, n)) * 0.1)

f1_0 = cl.F1([x11_f1, x12_f1])

e1s_F1, e2s_F1 = [], []
for eps in epss:
    f1_e   = cl.F1([x11_f1 + eps*dx11_f1, x12_f1 + eps*dx12_f1])
    _, df1 = cl.dF1([x11_f1, x12_f1], [dx11_f1, dx12_f1])
    d2f1   = cl.d2F_dF1([x11_f1, x12_f1], [dx11_f1, dx12_f1], [dx11_f1, dx12_f1], [None, None])

    res1 = f1_e - f1_0 - eps * df1
    res2 = res1 - 0.5 * eps**2 * d2f1
    e1s_F1.append(float(cp.linalg.norm(res1)))
    e2s_F1.append(float(cp.linalg.norm(res2)))

e1s_F1, e2s_F1 = np.array(e1s_F1), np.array(e2s_F1)

## F0:  $x_0 \mapsto \frac{1}{N}\||x_0| - d\|_2^2$

In [ ]:
x0_f0  = cp.array(rc((ntheta, ndist, nz, n)))
dx0_f0 = cp.array(rc((ntheta, ndist, nz, n)) * 0.1)

f0_0 = float(cl.F0(x0_f0, data_g))

e1s_F0, e2s_F0 = [], []
for eps in epss:
    f0_e = float(cl.F0(x0_f0 + eps*dx0_f0, data_g))
    df0  = float(cl.dF0(x0_f0, dx0_f0, data_g))
    d2f0 = float(cl.d2F_dF0(x0_f0, dx0_f0, dx0_f0, None, data_g))

    res1 = f0_e - f0_0 - eps * df0
    res2 = res1 - 0.5 * eps**2 * d2f0
    e1s_F0.append(abs(res1))
    e2s_F0.append(abs(res2))

e1s_F0, e2s_F0 = np.array(e1s_F0), np.array(e2s_F0)

In [ ]:
# Print convergence tables
def print_test(name, epss, e1s, e2s):
    print(f"\n{'='*68}")
    print(f"  {name}")
    print(f"  {'eps':>8}  {'e1':>12}  {'ratio':>7}  {'e2':>12}  {'ratio':>7}")
    for i, (eps, e1, e2) in enumerate(zip(epss, e1s, e2s)):
        r1 = f"{e1s[i-1]/e1:7.2f}" if i > 0 else "      -"
        r2 = f"{e2s[i-1]/e2:7.2f}" if i > 0 else "      -"
        print(f"  {eps:8.4f}  {e1:12.3e}  {r1}  {e2:12.3e}  {r2}")
    print(f"  Expected: e1 ratio ~4 (O(eps^2)), e2 ratio ~8 (O(eps^3))")

print_test("F3: S_{pos}(proj)  [second component]", epss, e1s_F3, e2s_F3)
print_test("F2: exp(i*x22)     [second component]", epss, e1s_F2, e2s_F2)
print_test("F1: D(x11*x12)     [bilinear -> e2=0]", epss, e1s_F1, e2s_F1)
print_test("F0: ||x|-d||^2",                        epss, e1s_F0, e2s_F0)

In [ ]:
# Convergence plots
tests = [
    ("F3: $\\mathcal{S}_{\\mathrm{pos}}(\\mathrm{proj})$", e1s_F3, e2s_F3),
    ("F2: $e^{i x_{22}}$",                                 e1s_F2, e2s_F2),
    ("F1: $D(x_{11} x_{12})$  [bilinear]",                 e1s_F1, e2s_F1),
    ("F0: $\\||x|-d\\|_2^2$",                               e1s_F0, e2s_F0),
]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, (title, e1s, e2s) in zip(axes, tests):
    # clip zeros for log scale
    e1s_plot = np.maximum(e1s, 1e-20)
    e2s_plot = np.maximum(e2s, 1e-20)

    ax.loglog(epss, e1s_plot, 'r-o', lw=2, ms=7, label='$e_1$ (1st-order residual)')
    ax.loglog(epss, e2s_plot, 'b-s', lw=2, ms=7, label='$e_2$ (2nd-order residual)')

    # reference slopes anchored at the first point
    ref_e  = epss / epss[0]
    ax.loglog(epss, e1s_plot[0] * ref_e**2, 'r--', alpha=0.45, lw=1.5, label='$O(\\varepsilon^2)$')
    ax.loglog(epss, e2s_plot[0] * ref_e**3, 'b--', alpha=0.45, lw=1.5, label='$O(\\varepsilon^3)$')

    ax.set_xlabel('$\\varepsilon$', fontsize=12)
    ax.set_ylabel('Error (L2 norm)', fontsize=10)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(True, which='both', alpha=0.3)

fig.suptitle(
    '$f(x+\\varepsilon\\,dx) = f(x) + \\varepsilon\\,df + '
    '\\frac{1}{2}\\varepsilon^2\\,d^2f + O(\\varepsilon^3)$,\\quad '
    '$d^2f = \\mathrm{d2F\\_dF}(x, dx, dx,\\, w{=}\\mathrm{None})$',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.show()